# Task 2 · Operation Inversion: starter

Every cell already runs (it submits all zeros, score ≈ 0). The `TODO`s mark what you have to write.

**Task.** A frozen neural network, saved as TorchScript in `model.pt`, maps 10 numbers `x0 … x9` to one number `y`. `data.csv` has 1000 rows of `x0 … x9` and the model's output `y`. Then one of the ten input columns was overwritten with `0.0`. Recover that column.

- Changing the model's weights is forbidden. You may only use it: forward passes and gradients.
- Metric: the MSE between your answer and the true column. Score = (MSE_baseline − MSE) / (MSE_baseline − MSE_best) · 100, clipped to [0, 100], with MSE_baseline ≈ 0.968 (answering all zeros) and MSE_best ≈ 0.0045.
- Submit `submission.csv` with the columns `subtaskID` (always 1), `datapointID` (0 … 999) and `answer`, together with your code.

> **Practice data.** `data/inversion_practice/` is a replica built by the coach. It is **not** the real task data: the real model and the real hidden column are different. Only the practice folder has `true_column.csv`, so you can score yourself.

In [ ]:
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(1)  # thousands of tiny forward passes: one thread is fastest, and the machine is shared

DATA_DIR = "../data/inversion_practice"  # Colab: upload model.pt, data.csv, true_column.csv and use "."

In [ ]:
# the loading code given in the task (only the paths are prefixed with DATA_DIR);
# newer torch versions warn that torch.jit is deprecated: it still works
df = pd.read_csv(f"{DATA_DIR}/data.csv")
y = df["y"].values.astype("float32")
X = df.drop(columns="y").values.astype("float32")
model = torch.jit.load(f"{DATA_DIR}/model.pt")
model.eval()
X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)
with torch.no_grad():
    pred = model(X_t).squeeze()
print("X", tuple(X_t.shape), " y", tuple(y_t.shape), " model output", tuple(pred.shape))
print(f"MSE between y and the model's output with the zeroed column: {((pred - y_t) ** 2).mean():.3f}")

In [ ]:
# Freeze the weights. This changes no weight; autograd just stops tracking them. Without it, a TorchScript
# model whose first call ran with gradients on keeps building graphs even inside torch.no_grad() (wasted
# time and memory), and torch.inference_mode() crashes. Gradients with respect to the INPUT still work.
for p in model.parameters():
    p.requires_grad_(False)

## Which column is missing?

In [ ]:
zero_cols = (X_t == 0).all(dim=0)  # True for a column that is exactly 0.0 in every row
k = int(zero_cols.nonzero()[0])
N = len(X_t)
print(f"all-zero columns: {zero_cols.nonzero().flatten().tolist()}  ->  the missing column is x{k}")

## The model as a function of the missing column

`predict(z)` puts your guess `z` (one number per row) into column k and runs the model. Training changed the weights to fit the data; here the weights are fixed and **the input** is what we change: the same backprop, a different leaf.

In [ ]:
def predict(z):
    # model output (N,) when column k is replaced by z (N,); the other 9 columns stay exactly as given
    X_z = torch.cat([X_t[:, :k], z[:, None], X_t[:, k + 1:]], dim=1)
    return model(X_z)[:, 0]  # (N, 1) -> (N,): (N, 1) - (N,) would silently broadcast to (N, N)

## Gradient descent on the missing column only

Fill in the loop. Questions to answer as you go:
- Which optimizer and learning rate? Remember what an Adam step does, and that the missing values are about N(0, 1).
- Sum or mean over the 1000 rows? Does the answer change between SGD and Adam?
- Starting point: 0 is the natural guess. Is it always a good one?

In [ ]:
z = torch.zeros(N, requires_grad=True)  # our guess for the missing column, one unknown per row
opt = None  # TODO: an optimizer that receives ONLY [z]  (never model.parameters())
for step in range(0):  # TODO: how many steps?
    # TODO: loss = the squared residuals (predict(z) - y_t) ** 2, summed or averaged over the rows
    # TODO: the beats from the training loop: zero_grad -> backward -> step
    pass
answer = z.detach().numpy().copy()

## Residual check: which rows are not solved?

A row is solved when `predict(answer)` reproduces its `y`. Careful: a small residual does not prove the answer is right. Some rows have **several** values that give exactly the same `y`.

In [ ]:
with torch.no_grad():
    residual = (predict(torch.from_numpy(answer)) - y_t).abs()
print(f"median |residual| {residual.median():.1e}; rows with |residual| > 1e-3: {(residual > 1e-3).sum().item()} of {N}")
worst = residual.argsort(descending=True)[:5].numpy()
pd.DataFrame({"row": worst, "answer": answer[worst], "|residual|": residual[worst].numpy()})

## Look at one row

There is only one unknown per row, so you can simply try 2001 values and draw the curve. Every crossing of the dashed line is an exact solution. Try several rows: how many crossings do you see, and which one would you submit?

TODO (to beat plain gradient descent): do this for **all** rows at once, find every crossing, and choose between them with a prior. Are the other columns correlated with the missing one?

In [ ]:
import matplotlib.pyplot as plt

row = int(worst[0])  # try other rows too
grid = torch.linspace(-5, 5, 2001)
X_row = X_t[row].repeat(len(grid), 1)  # the same row 2001 times ...
X_row[:, k] = grid  # ... with 2001 different values in the missing column
with torch.no_grad():
    curve = model(X_row)[:, 0]

plt.figure(figsize=(6, 3.5))
plt.plot(grid, curve, label=f"model output, row {row}")
plt.axhline(y_t[row], color="black", linestyle="--", label="target y")
plt.axvline(answer[row], color="tab:red", label="your answer")
plt.xlabel(f"value put into x{k}")
plt.legend()
plt.show()

## Self-score (practice data only)

The real task has no `true_column.csv`; this is the scoring formula applied to the practice replica.

In [ ]:
truth = pd.read_csv(f"{DATA_DIR}/true_column.csv")["x_true"].values
mse = np.mean((answer - truth) ** 2)
score = np.clip((0.968 - mse) / (0.968 - 0.0045) * 100, 0, 100)
print(f"MSE {mse:.4f}   score {score:.1f} / 100   (all zeros on this replica: MSE {np.mean(truth ** 2):.4f})")

## Write the submission

In [ ]:
sub = pd.DataFrame({"subtaskID": 1, "datapointID": range(1000), "answer": answer})
sub.to_csv("submission.csv", index=False)
sub.head()